#Session 8 | Part 2

Start Spark:

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Session08Part02")
    .master("local[*]")
    .getOrCreate()
)

Create an analytics dataset

In [2]:
orders = [
    (1, "Athens", "Laptop", "Electronics", 1, 1200.00),
    (2, "Athens", "Mouse", "Electronics", 2, 25.00),
    (3, "Thessaloniki", "Desk", "Furniture", 1, 240.00),
    (4, "Patras", "Chair", "Furniture", 4, 85.00),
    (5, "Athens", "Notebook", "Office", 10, 3.50),
    (6, "Heraklion", "Monitor", "Electronics", 2, 310.00),
    (7, "Patras", "Pen Pack", "Office", 5, 6.00),
    (8, "Thessaloniki", "Keyboard", "Electronics", 1, 75.00),
    (9, "Athens", "Desk Lamp", "Furniture", 3, 32.00),
    (10, "Heraklion", "Notebook", "Office", 20, 3.25),
]

columns = ["order_id", "city", "product", "category", "quantity", "unit_price"]

orders_df = spark.createDataFrame(orders, columns)
orders_df.show()

+--------+------------+---------+-----------+--------+----------+
|order_id|        city|  product|   category|quantity|unit_price|
+--------+------------+---------+-----------+--------+----------+
|       1|      Athens|   Laptop|Electronics|       1|    1200.0|
|       2|      Athens|    Mouse|Electronics|       2|      25.0|
|       3|Thessaloniki|     Desk|  Furniture|       1|     240.0|
|       4|      Patras|    Chair|  Furniture|       4|      85.0|
|       5|      Athens| Notebook|     Office|      10|       3.5|
|       6|   Heraklion|  Monitor|Electronics|       2|     310.0|
|       7|      Patras| Pen Pack|     Office|       5|       6.0|
|       8|Thessaloniki| Keyboard|Electronics|       1|      75.0|
|       9|      Athens|Desk Lamp|  Furniture|       3|      32.0|
|      10|   Heraklion| Notebook|     Office|      20|      3.25|
+--------+------------+---------+-----------+--------+----------+



Add a revenue column:

In [3]:
from pyspark.sql.functions import col

orders_df = orders_df.withColumn("revenue", col("quantity") * col("unit_price"))
orders_df.show()

+--------+------------+---------+-----------+--------+----------+-------+
|order_id|        city|  product|   category|quantity|unit_price|revenue|
+--------+------------+---------+-----------+--------+----------+-------+
|       1|      Athens|   Laptop|Electronics|       1|    1200.0| 1200.0|
|       2|      Athens|    Mouse|Electronics|       2|      25.0|   50.0|
|       3|Thessaloniki|     Desk|  Furniture|       1|     240.0|  240.0|
|       4|      Patras|    Chair|  Furniture|       4|      85.0|  340.0|
|       5|      Athens| Notebook|     Office|      10|       3.5|   35.0|
|       6|   Heraklion|  Monitor|Electronics|       2|     310.0|  620.0|
|       7|      Patras| Pen Pack|     Office|       5|       6.0|   30.0|
|       8|Thessaloniki| Keyboard|Electronics|       1|      75.0|   75.0|
|       9|      Athens|Desk Lamp|  Furniture|       3|      32.0|   96.0|
|      10|   Heraklion| Notebook|     Office|      20|      3.25|   65.0|
+--------+------------+---------+-----

 Register a SQL view

In [4]:
orders_df.createOrReplaceTempView("orders")

In [5]:
spark.sql("""
    SELECT *
    FROM orders
""").show()

+--------+------------+---------+-----------+--------+----------+-------+
|order_id|        city|  product|   category|quantity|unit_price|revenue|
+--------+------------+---------+-----------+--------+----------+-------+
|       1|      Athens|   Laptop|Electronics|       1|    1200.0| 1200.0|
|       2|      Athens|    Mouse|Electronics|       2|      25.0|   50.0|
|       3|Thessaloniki|     Desk|  Furniture|       1|     240.0|  240.0|
|       4|      Patras|    Chair|  Furniture|       4|      85.0|  340.0|
|       5|      Athens| Notebook|     Office|      10|       3.5|   35.0|
|       6|   Heraklion|  Monitor|Electronics|       2|     310.0|  620.0|
|       7|      Patras| Pen Pack|     Office|       5|       6.0|   30.0|
|       8|Thessaloniki| Keyboard|Electronics|       1|      75.0|   75.0|
|       9|      Athens|Desk Lamp|  Furniture|       3|      32.0|   96.0|
|      10|   Heraklion| Notebook|     Office|      20|      3.25|   65.0|
+--------+------------+---------+-----

 Filter with SQL

In [6]:
spark.sql("""
    SELECT order_id, city, product, revenue
    FROM orders
    WHERE revenue >= 100
""").show()

+--------+------------+-------+-------+
|order_id|        city|product|revenue|
+--------+------------+-------+-------+
|       1|      Athens| Laptop| 1200.0|
|       3|Thessaloniki|   Desk|  240.0|
|       4|      Patras|  Chair|  340.0|
|       6|   Heraklion|Monitor|  620.0|
+--------+------------+-------+-------+



Show only orders from Athens

In [9]:
spark.sql("""
    SELECT order_id, city, product, revenue
    FROM orders
    WHERE city = "Athens"
""").show()

+--------+------+---------+-------+
|order_id|  city|  product|revenue|
+--------+------+---------+-------+
|       1|Athens|   Laptop| 1200.0|
|       2|Athens|    Mouse|   50.0|
|       5|Athens| Notebook|   35.0|
|       9|Athens|Desk Lamp|   96.0|
+--------+------+---------+-------+



Group with SQL

In [10]:
spark.sql("""
    SELECT
        category,
        COUNT(*) AS order_count,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM orders
    GROUP BY category
    ORDER BY total_revenue DESC
""").show()

+-----------+-----------+-------------+
|   category|order_count|total_revenue|
+-----------+-----------+-------------+
|Electronics|          4|       1945.0|
|  Furniture|          3|        676.0|
|     Office|          3|        130.0|
+-----------+-----------+-------------+



Group by city and show total revenue per city.

In [12]:
spark.sql("""
    SELECT
        city,
        COUNT(*) AS order_count,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM orders
    GROUP BY city
    ORDER BY total_revenue DESC
""").show()

+------------+-----------+-------------+
|        city|order_count|total_revenue|
+------------+-----------+-------------+
|      Athens|          4|       1381.0|
|   Heraklion|          2|        685.0|
|      Patras|          2|        370.0|
|Thessaloniki|          2|        315.0|
+------------+-----------+-------------+



Compare SQL and DataFrame syntax

SQL version:

In [13]:
spark.sql("""
    SELECT category, AVG(revenue) AS average_revenue
    FROM orders
    GROUP BY category
""").show()

+-----------+------------------+
|   category|   average_revenue|
+-----------+------------------+
|     Office|43.333333333333336|
|Electronics|            486.25|
|  Furniture|225.33333333333334|
+-----------+------------------+



In [14]:
from pyspark.sql.functions import avg

orders_df.groupBy("category").agg(
    avg("revenue").alias("average_revenue")
).show()

+-----------+------------------+
|   category|   average_revenue|
+-----------+------------------+
|     Office|43.333333333333336|
|Electronics|            486.25|
|  Furniture|225.33333333333334|
+-----------+------------------+



## Exercise 2

Create the orders_df DataFrame.

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, sum

In [16]:
spark = (
    SparkSession.builder
    .appName("Session08Exercise02")
    .master("local[*]")
    .getOrCreate()
)

In [18]:
orders = [
    (1, "Athens", "Laptop", "Electronics", 5, 1200.00),
    (2, "Athens", "Mouse", "Electronics", 40, 25.00),
    (3, "Thessaloniki", "Desk", "Furniture", 10, 240.00),
    (4, "Patras", "Chair", "Furniture", 25, 85.00),
    (5, "Athens", "Notebook", "Office", 47, 3.50),
    (6, "Heraklion", "Monitor", "Electronics", 100, 310.00),
    (7, "Patras", "Pen Pack", "Office", 69, 6.00),
    (8, "Thessaloniki", "Keyboard", "Electronics", 83, 75.00),
    (9, "Athens", "Desk Lamp", "Furniture", 35, 32.00),
    (10, "Heraklion", "Notebook", "Office", 72, 3.25),
]

columns = ["order_id", "city", "product", "category", "quantity", "unit_price"]

orders_df = spark.createDataFrame(orders, columns)
orders_df.show()

+--------+------------+---------+-----------+--------+----------+
|order_id|        city|  product|   category|quantity|unit_price|
+--------+------------+---------+-----------+--------+----------+
|       1|      Athens|   Laptop|Electronics|       5|    1200.0|
|       2|      Athens|    Mouse|Electronics|      40|      25.0|
|       3|Thessaloniki|     Desk|  Furniture|      10|     240.0|
|       4|      Patras|    Chair|  Furniture|      25|      85.0|
|       5|      Athens| Notebook|     Office|      47|       3.5|
|       6|   Heraklion|  Monitor|Electronics|     100|     310.0|
|       7|      Patras| Pen Pack|     Office|      69|       6.0|
|       8|Thessaloniki| Keyboard|Electronics|      83|      75.0|
|       9|      Athens|Desk Lamp|  Furniture|      35|      32.0|
|      10|   Heraklion| Notebook|     Office|      72|      3.25|
+--------+------------+---------+-----------+--------+----------+



Add the revenue column

In [19]:
orders_df = orders_df.withColumn(
    "revenue",
    col("quantity") * col("unit_price")
)

orders_df.show()

+--------+------------+---------+-----------+--------+----------+-------+
|order_id|        city|  product|   category|quantity|unit_price|revenue|
+--------+------------+---------+-----------+--------+----------+-------+
|       1|      Athens|   Laptop|Electronics|       5|    1200.0| 6000.0|
|       2|      Athens|    Mouse|Electronics|      40|      25.0| 1000.0|
|       3|Thessaloniki|     Desk|  Furniture|      10|     240.0| 2400.0|
|       4|      Patras|    Chair|  Furniture|      25|      85.0| 2125.0|
|       5|      Athens| Notebook|     Office|      47|       3.5|  164.5|
|       6|   Heraklion|  Monitor|Electronics|     100|     310.0|31000.0|
|       7|      Patras| Pen Pack|     Office|      69|       6.0|  414.0|
|       8|Thessaloniki| Keyboard|Electronics|      83|      75.0| 6225.0|
|       9|      Athens|Desk Lamp|  Furniture|      35|      32.0| 1120.0|
|      10|   Heraklion| Notebook|     Office|      72|      3.25|  234.0|
+--------+------------+---------+-----

Register the DataFrame as a temporary view named orders

In [20]:
orders_df.createOrReplaceTempView("orders")

Write SQL queries that answer

Which city has the highest total revenue?


In [30]:
spark.sql("""
    SELECT city, SUM(revenue) AS total_revenue
    FROM orders
    GROUP BY city
    ORDER BY total_revenue DESC
""").show()

+------------+-------------+
|        city|total_revenue|
+------------+-------------+
|   Heraklion|      31234.0|
|Thessaloniki|       8625.0|
|      Athens|       8284.5|
|      Patras|       2539.0|
+------------+-------------+



Equivalent DataFrame:

In [35]:
orders_df.groupBy("city").agg(sum("revenue").alias("total_revenue")).show()

+------------+-------------+
|        city|total_revenue|
+------------+-------------+
|Thessaloniki|       8625.0|
|      Athens|       8284.5|
|      Patras|       2539.0|
|   Heraklion|      31234.0|
+------------+-------------+



Which category has the most orders?

In [31]:
spark.sql("""
    SELECT category, SUM(quantity) AS total_orders
    FROM orders
    GROUP BY category
    ORDER BY total_orders DESC
""").show()

+-----------+------------+
|   category|total_orders|
+-----------+------------+
|Electronics|         228|
|     Office|         188|
|  Furniture|          70|
+-----------+------------+



Equivalent DataFrame:

In [36]:
orders_df.groupBy("category").agg(sum("quantity").alias("total_orders")).show()

+-----------+------------+
|   category|total_orders|
+-----------+------------+
|     Office|         188|
|Electronics|         228|
|  Furniture|          70|
+-----------+------------+



Which products have revenue greater than 2000?

In [32]:
spark.sql("""
    SELECT *
    FROM orders
    WHERE revenue >= 2000
""").show()

+--------+------------+--------+-----------+--------+----------+-------+
|order_id|        city| product|   category|quantity|unit_price|revenue|
+--------+------------+--------+-----------+--------+----------+-------+
|       1|      Athens|  Laptop|Electronics|       5|    1200.0| 6000.0|
|       3|Thessaloniki|    Desk|  Furniture|      10|     240.0| 2400.0|
|       4|      Patras|   Chair|  Furniture|      25|      85.0| 2125.0|
|       6|   Heraklion| Monitor|Electronics|     100|     310.0|31000.0|
|       8|Thessaloniki|Keyboard|Electronics|      83|      75.0| 6225.0|
+--------+------------+--------+-----------+--------+----------+-------+



Equivalent DataFrame:

In [37]:
orders_df[orders_df["revenue"] >= 2000].show()

+--------+------------+--------+-----------+--------+----------+-------+
|order_id|        city| product|   category|quantity|unit_price|revenue|
+--------+------------+--------+-----------+--------+----------+-------+
|       1|      Athens|  Laptop|Electronics|       5|    1200.0| 6000.0|
|       3|Thessaloniki|    Desk|  Furniture|      10|     240.0| 2400.0|
|       4|      Patras|   Chair|  Furniture|      25|      85.0| 2125.0|
|       6|   Heraklion| Monitor|Electronics|     100|     310.0|31000.0|
|       8|Thessaloniki|Keyboard|Electronics|      83|      75.0| 6225.0|
+--------+------------+--------+-----------+--------+----------+-------+



What is the average unit price per category?

In [34]:
spark.sql("""
    SELECT category, SUM(unit_price)/COUNT(category) AS avg_price_per_unit
    FROM orders
    GROUP BY category
""").show()

+-----------+-----------+
|   category|total_price|
+-----------+-----------+
|     Office|       4.25|
|Electronics|      402.5|
|  Furniture|      119.0|
+-----------+-----------+



Equivalent DataFrame:

In [39]:
orders_df.groupBy("category").agg(avg("unit_price").alias("avg_price_per_unit")).show()

+-----------+------------------+
|   category|avg_price_per_unit|
+-----------+------------------+
|     Office|              4.25|
|Electronics|             402.5|
|  Furniture|             119.0|
+-----------+------------------+

